# 04 — Analisis Domain Shift, 5 Kasus Kegagalan, & Eksperimen Uji Hipotesis
**Runtime:** GPU T4 (atau CPU jika hanya membuat grafik rekap).

Notebook ini membantu menghasilkan seluruh grafik dan bukti eksperimen kecil untuk laporan:
1. **Tabel & Grafik Domain Shift**: Perbandingan performa 3 model di `Test_Asli` vs `Test_HP_Sendiri` serta selisih (Δ mAP).
2. **Grafik Eksperimen NRP 0**: Pengaruh ukuran input `320` vs `640` vs `960`.
3. **Visualisasi 5 Foto Gagal** dari Test Set HP Anda.
4. **Eksperimen Kecil Uji Hipotesis Kegagalan**:
   - Uji A (Mekanisme resolusi prototype mask YOLO): Bandingkan prediksi foto yang sama di `imgsz=320` vs `640` vs `960`.
   - Uji B (Kondisi *low-light / backlight*): Terapkan **CLAHE** (peningkatan kontras adaptif) pada foto gagal lalu inferensi ulang untuk membuktikan apakah kegagalan disebabkan oleh kontras pencahayaan.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics opencv-python-headless matplotlib pandas

import os, glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

PROJECT_DIR = '/content/drive/MyDrive/CPMK3_Instance_Segmentation'
LOGS_DIR    = os.path.join(PROJECT_DIR, 'logs')
EXPORTS_DIR = os.path.join(PROJECT_DIR, 'exports')
os.makedirs(EXPORTS_DIR, exist_ok=True)

dfs = []
for fname in ['eval_yolov8_seg.csv', 'eval_maskrcnn.csv', 'eval_mask2former.csv']:
    p = os.path.join(LOGS_DIR, fname)
    if os.path.exists(p):
        dfs.append(pd.read_csv(p))

df_all = pd.concat(dfs, ignore_index=True)
df_all.to_csv(os.path.join(EXPORTS_DIR, 'summary_all_models.csv'), index=False)
df_all


## 1. Analisis Domain Shift (Selisih Test Asli vs Test HP Sendiri)


In [ ]:
# Fokus perbandingan antar 3 model di ukuran baseline 640
df_640 = df_all[df_all['imgsz'] == 640]
pivot = df_640.pivot(index=['model', 'architecture_family'], columns='dataset', values='mask_mAP50')
if 'Test_Asli' in pivot.columns and 'Test_HP_Sendiri' in pivot.columns:
    pivot['Delta_Domain_Shift'] = (pivot['Test_HP_Sendiri'] - pivot['Test_Asli']).round(4)
print(pivot)

ax = pivot[['Test_Asli', 'Test_HP_Sendiri']].plot(kind='bar', figsize=(9, 5))
plt.title('Perbandingan Mask mAP@50 (imgsz=640): Test Asli vs Test HP Sendiri')
plt.ylabel('Mask mAP@50')
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(EXPORTS_DIR, 'chart_domain_shift_640.png'), dpi=150)
plt.show()


## 2. Grafik Eksperimen NRP 0 (Ukuran Input: 320 vs 640 vs 960)


In [ ]:
df_yolo = df_all[df_all['model'] == 'YOLOv8s-seg']
plt.figure(figsize=(8, 5))
for ds_name, grp in df_yolo.groupby('dataset'):
    grp = grp.sort_values('imgsz')
    plt.plot(grp['imgsz'], grp['mask_mAP50'], marker='o', linewidth=2, label=f'{ds_name} (Mask mAP@50)')
    plt.plot(grp['imgsz'], grp['mask_mAP50_95'], marker='s', linestyle='--', label=f'{ds_name} (Mask mAP@50-95)')

plt.xticks([320, 640, 960])
plt.xlabel('Ukuran Input (px) — Parameter NRP 0')
plt.ylabel('Score')
plt.title('Pengaruh Ukuran Input (320 / 640 / 960) pada YOLOv8s-seg')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(EXPORTS_DIR, 'chart_nrp0_input_size.png'), dpi=150)
plt.show()


## 3. Eksperimen Kecil Uji Hipotesis Kegagalan (5 Foto Gagal)
Masukkan nama file **5 foto gagal** dari folder test set HP Anda di daftar `FAILED_IMAGES` di bawah ini.
Skrip akan:
1. Membandingkan prediksi **320 vs 640 vs 960** (menguji hipotesis resolusi prototype mask YOLO yang berukuran 1/4 input: 80×80 vs 160×160 vs 240×240).
2. Membandingkan **foto asli vs foto setelah CLAHE** (menguji hipotesis kegagalan akibat cahaya redup / *backlight*).


In [ ]:
HP_IMG_DIR = os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_yolo', 'test', 'images')
if not os.path.exists(HP_IMG_DIR):
    HP_IMG_DIR = os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_yolo', 'train', 'images')

# Tampilkan 10 file pertama agar mudah memilih 5 foto gagal
sample_files = sorted(os.listdir(HP_IMG_DIR))
print('Contoh nama file di Test HP:', sample_files[:10])

# === GANTI DENGAN 5 NAMA FILE FOTO GAGAL ANDA ===
FAILED_IMAGES = sample_files[:5]

def apply_clahe(bgr_img):
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

models_yolo = {}
for sz in [320, 640, 960]:
    pt = os.path.join(PROJECT_DIR, 'runs', 'yolov8_seg', f'yolov8s_seg_img{sz}', 'weights', 'best.pt')
    if os.path.exists(pt):
        models_yolo[sz] = YOLO(pt)

for idx, fname in enumerate(FAILED_IMAGES, 1):
    fpath = os.path.join(HP_IMG_DIR, fname)
    if not os.path.exists(fpath):
        continue
    img_bgr = cv2.imread(fpath)
    img_clahe = apply_clahe(img_bgr)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    # Panel 1-3: Uji Resolusi (320 / 640 / 960)
    for ax_i, sz in enumerate([320, 640, 960]):
        if sz in models_yolo:
            res = models_yolo[sz].predict(img_bgr, imgsz=sz, conf=0.25, verbose=False)[0]
            vis = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
            axes[ax_i].imshow(vis)
            axes[ax_i].set_title(f'YOLOv8s-seg ({sz}px, proto {sz//4}x{sz//4})')
        axes[ax_i].axis('off')

    # Panel 4: Uji CLAHE pada 640px (untuk kasus low-light / backlight)
    if 640 in models_yolo:
        res_c = models_yolo[640].predict(img_clahe, imgsz=640, conf=0.25, verbose=False)[0]
        vis_c = cv2.cvtColor(res_c.plot(), cv2.COLOR_BGR2RGB)
        axes[3].imshow(vis_c)
        axes[3].set_title('Uji Hipotesis: CLAHE + 640px')
    axes[3].axis('off')

    plt.suptitle(f'Kasus Kegagalan #{idx}: {fname}', fontsize=14)
    plt.tight_layout()
    out_fig = os.path.join(EXPORTS_DIR, f'failure_case_{idx}_{os.path.splitext(fname)[0]}.png')
    plt.savefig(out_fig, dpi=150)
    plt.show()
    print('Disimpan:', out_fig)
